# core

> Alignment state context, getters, and updaters for route handlers

In [ ]:
#| default_exp routes.alignment.core

In [ ]:
#| export
from typing import Any, Dict, List, Optional, NamedTuple

from cjm_fasthtml_interactions.core.state_store import get_session_id
from cjm_fasthtml_card_stack.core.models import CardStackState
from cjm_workflow_state.history import push_history, pop_history

from cjm_fasthtml_workflow_transcript_decomp.core.models import (
    VADChunk, WorkingSegment, AlignmentStepState
)

# Debug flag for alignment state tracing (set False in production)
DEBUG_ALIGN_STATE = False

## AlignContext

Common state values loaded by alignment handlers in a single call.

In [ ]:
#| export
class AlignContext(NamedTuple):
    """Common alignment state values loaded by handlers."""
    chunk_dicts: List[Dict[str, Any]]  # Serialized VAD chunks
    focused_chunk_index: int  # Currently focused VAD chunk index
    visible_count: int  # Number of visible cards in viewport
    card_width: int  # Card stack width in rem
    history: list  # Undo history stack
    media_path: Optional[str]  # Path to audio file
    audio_duration: Optional[float]  # Total audio duration

## State Management

Functions for reading and writing alignment state via the workflow state store.

In [ ]:
#| export
def _get_alignment_state(
    workflow:Any,  # StructureDecompWorkflow instance
    session_id:str,  # Session identifier
) -> AlignmentStepState:  # Typed alignment step state
    """Get the alignment step state from the workflow state store."""
    state = workflow.state_store.get_state(workflow.config.workflow_id, session_id)
    step_states = state.get("step_states", {})
    return step_states.get("alignment", {})

def _load_alignment_context(
    workflow:Any,  # StructureDecompWorkflow instance
    session_id:str,  # Session identifier
) -> AlignContext:  # Loaded context with all common alignment state
    """Load commonly-needed alignment state values in a single call."""
    state = _get_alignment_state(workflow, session_id)
    return AlignContext(
        chunk_dicts=state.get("vad_chunks", []),
        focused_chunk_index=state.get("focused_chunk_index", 0),
        visible_count=state.get("visible_count", 5),
        card_width=state.get("card_width", 40),
        history=state.get("history", []),
        media_path=state.get("media_path"),
        audio_duration=state.get("audio_duration"),
    )

def _update_alignment_state(
    workflow:Any,  # StructureDecompWorkflow instance
    session_id:str,  # Session identifier
    vad_chunks=None,  # Updated VAD chunks (serialized)
    focused_chunk_index=None,  # Updated focused chunk index
    is_initialized=None,  # Initialization flag
    history=None,  # Updated history
    visible_count=None,  # Visible card count
    card_width=None,  # Card stack width in rem
    media_path=None,  # Audio file path
    audio_duration=None,  # Audio duration
) -> None:
    """Update the alignment step state in the workflow state store."""
    if DEBUG_ALIGN_STATE:
        print(f"[ALIGN_STATE] _update_alignment_state called")

    # Read full state to preserve sibling step states
    workflow_state = workflow.state_store.get_state(workflow.config.workflow_id, session_id)

    if DEBUG_ALIGN_STATE:
        print(f"[ALIGN_STATE] BEFORE: step_states keys = {list(workflow_state.get('step_states', {}).keys())}")
        selection_state = workflow_state.get('step_states', {}).get('selection', {})
        print(f"[ALIGN_STATE] BEFORE: selection.selected_sources count = {len(selection_state.get('selected_sources', []))}")

    step_states = workflow_state.get("step_states", {})
    align_state = step_states.get("alignment", {})

    # Update only provided fields
    if vad_chunks is not None:
        align_state["vad_chunks"] = vad_chunks
    if focused_chunk_index is not None:
        align_state["focused_chunk_index"] = focused_chunk_index
    if is_initialized is not None:
        align_state["is_initialized"] = is_initialized
    if history is not None:
        align_state["history"] = history
    if visible_count is not None:
        align_state["visible_count"] = visible_count
    if card_width is not None:
        align_state["card_width"] = card_width
    if media_path is not None:
        align_state["media_path"] = media_path
    if audio_duration is not None:
        align_state["audio_duration"] = audio_duration

    step_states["alignment"] = align_state
    workflow_state["step_states"] = step_states

    if DEBUG_ALIGN_STATE:
        print(f"[ALIGN_STATE] AFTER: step_states keys = {list(step_states.keys())}")

    workflow.state_store.update_state(workflow.config.workflow_id, session_id, workflow_state)

    if DEBUG_ALIGN_STATE:
        verify_state = workflow.state_store.get_state(workflow.config.workflow_id, session_id)
        print(f"[ALIGN_STATE] VERIFY: step_states keys = {list(verify_state.get('step_states', {}).keys())}")

## History Helpers

In [ ]:
#| export
def _push_alignment_history(
    workflow:Any,  # StructureDecompWorkflow instance
    session_id:str,  # Session identifier
    current_chunks:List[Dict[str, Any]],  # Current chunk state to snapshot
    focused_index:int,  # Current focused index to snapshot
) -> int:  # New history depth after push
    """Push current alignment state to history stack before making changes."""
    state = _get_alignment_state(workflow, session_id)
    history = state.get("history", [])

    snapshot = {"vad_chunks": current_chunks, "focused_chunk_index": focused_index}
    new_history = push_history(
        history, snapshot,
        max_depth=workflow.config.max_history_depth
    )

    _update_alignment_state(workflow, session_id, history=new_history)
    return len(new_history)

## Conversion Helpers

In [ ]:
#| export
def _to_vad_chunks(
    chunk_dicts:List[Dict[str, Any]]  # Serialized VAD chunk dictionaries
) -> List[VADChunk]:  # List of VADChunk objects
    """Convert chunk dictionaries to VADChunk objects."""
    return [VADChunk.from_dict(c) for c in chunk_dicts]

def _build_card_stack_state(
    ctx:AlignContext,  # Loaded alignment context
    active_mode:str=None,  # Current interaction mode name (unused for alignment)
) -> CardStackState:  # Card stack state for library functions
    """Build a CardStackState from alignment context for library functions."""
    return CardStackState(
        focused_index=ctx.focused_chunk_index,
        visible_count=ctx.visible_count,
        card_width=ctx.card_width,
        active_mode=active_mode,
    )

## Cross-State Helpers

Functions for reading and writing decomposition state from alignment handlers.
Assignment operations update both alignment (VADChunk) and decomposition (WorkingSegment) state.

In [ ]:
#| export
def _get_decomp_segments(
    workflow:Any,  # StructureDecompWorkflow instance
    session_id:str,  # Session identifier
) -> List[WorkingSegment]:  # List of WorkingSegment objects
    """Get decomposition segments from state."""
    state = workflow.state_store.get_state(workflow.config.workflow_id, session_id)
    step_states = state.get("step_states", {})
    decomp_state = step_states.get("decomposition", {})
    segment_dicts = decomp_state.get("segments", [])
    return [WorkingSegment.from_dict(s) for s in segment_dicts]

def _update_decomp_segments(
    workflow:Any,  # StructureDecompWorkflow instance
    session_id:str,  # Session identifier
    segments:List[WorkingSegment],  # Updated segments to save
) -> None:
    """Update decomposition segments in state."""
    # Read full state to preserve sibling step states
    workflow_state = workflow.state_store.get_state(workflow.config.workflow_id, session_id)
    step_states = workflow_state.get("step_states", {})
    decomp_state = step_states.get("decomposition", {})

    # Update segments
    decomp_state["segments"] = [s.to_dict() for s in segments]

    step_states["decomposition"] = decomp_state
    workflow_state["step_states"] = step_states
    workflow.state_store.update_state(workflow.config.workflow_id, session_id, workflow_state)

def _get_decomp_focused_index(
    workflow:Any,  # StructureDecompWorkflow instance
    session_id:str,  # Session identifier
) -> int:  # Currently focused segment index
    """Get the decomposition focused segment index."""
    state = workflow.state_store.get_state(workflow.config.workflow_id, session_id)
    step_states = state.get("step_states", {})
    decomp_state = step_states.get("decomposition", {})
    return decomp_state.get("focused_index", 0)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()